In [8]:
# import numpy as np
# from ultralytics import YOLO
# from pathlib import Path

# # ----------------------------
# # Model and dataset paths
# # ----------------------------
# model_path = "/home/nathaniel/Documents/GitHub/ClassificationTrainer/runs/pose/train10/weights/best.pt"
# images_dir = Path("/home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/images/test")
# labels_dir = Path("/home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/labels/test")
# # ----------------------------

# model = YOLO(model_path)

# all_errors = []
# missed_images = []

# for img_path in sorted(images_dir.glob("*")):

#     # Predict keypoints
#     result = model.predict(img_path, verbose=False)[0]

#     # Skip if no keypoints detected
#     if result.keypoints is None or len(result.keypoints.xy) == 0:
#         missed_images.append(img_path.name)
#         continue

#     pred = result.keypoints.xy[0].cpu().numpy()

#     # Load ground truth keypoints
#     label_path = labels_dir / (img_path.stem + ".txt")
#     try:
#         vals = np.loadtxt(label_path)
#     except Exception as e:
#         print(f"WARNING: could not load label for {img_path.name}: {e}")
#         missed_images.append(img_path.name)
#         continue

#     # Only one pig per image
#     if vals.ndim > 1:
#         vals = vals[0]

#     kpts = vals[5:].reshape(-1,3)[:,:2]  # ignore visibility

#     h, w = result.orig_shape

#     gt = kpts.copy()
#     gt[:,0] *= w
#     gt[:,1] *= h

#     # Distance per keypoint
#     dist = np.linalg.norm(pred - gt, axis=1)
#     all_errors.append(dist)

# # ============================

# all_errors = np.array(all_errors)

# print("\nImages evaluated:", len(all_errors))
# print("Images skipped (no detection or missing label):", len(missed_images))
# if missed_images:
#     print(missed_images)

# print("Mean Pixel Error:", all_errors.mean())
# print("RMSE:", np.sqrt((all_errors**2).mean()))
# print("\nPer-keypoint Mean Error:")
# keypoint_names = [
#     "Nose", "Ear_left", "Ear_right", "Spine1", "Shoulder_left", "Shoulder_right", 
#     "Center", "Spine2", "Hip_left", "Hip_right", "Tail_base"
# ]
# for name, e in zip(keypoint_names, all_errors.mean(axis=0)):
#     print(f"{name:15}: {e:.2f} px")


#################################################################################
# import numpy as np
# from ultralytics import YOLO
# from pathlib import Path

# # ----------------------------
# # Paths
# # ----------------------------
# model_path = "/home/nathaniel/Documents/GitHub/ClassificationTrainer/runs/pose/train10/weights/best.pt"
# images_base = Path("/home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/images")
# labels_base = Path("/home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/labels")
# # ----------------------------

# model = YOLO(model_path)

# keypoint_names = [
#     "Nose", "Ear_left", "Ear_right", "Spine1", "Shoulder_left", "Shoulder_right",
#     "Center", "Spine2", "Hip_left", "Hip_right", "Tail_base"
# ]

# # ----------------------------
# # Function to compute per-keypoint errors and RMSE
# # ----------------------------
# def compute_errors(split, cutoff_px=50):
#     images_dir = images_base / split
#     labels_dir = labels_base / split

#     all_errors = []
#     missed_images = []

#     for img_path in sorted(images_dir.glob("*")):

#         # Predict keypoints
#         result = model.predict(img_path, verbose=False)[0]

#         # Skip if no keypoints detected
#         if result.keypoints is None or len(result.keypoints.xy) == 0:
#             missed_images.append(img_path.name)
#             continue

#         pred = result.keypoints.xy[0].cpu().numpy()

#         # Load GT
#         label_path = labels_dir / (img_path.stem + ".txt")
#         try:
#             vals = np.loadtxt(label_path)
#         except Exception as e:
#             missed_images.append(img_path.name)
#             continue

#         if vals.ndim > 1:
#             vals = vals[0]

#         kpts = vals[5:].reshape(-1,3)[:,:2]  # ignore visibility

#         h, w = result.orig_shape
#         gt = kpts.copy()
#         gt[:,0] *= w
#         gt[:,1] *= h

#         dist = np.linalg.norm(pred - gt, axis=1)
#         all_errors.append(dist)

#     all_errors = np.array(all_errors)

#     mean_rmse = np.sqrt((all_errors**2).mean())
#     rmse_cutoff = np.sqrt((all_errors[all_errors < cutoff_px]**2).mean()) if np.any(all_errors < cutoff_px) else np.nan
#     per_keypoint_mean = all_errors.mean(axis=0)

#     return all_errors, mean_rmse, rmse_cutoff, per_keypoint_mean, missed_images

# # ----------------------------
# # Compute metrics for train
# # ----------------------------
# train_errors, train_rmse, train_rmse_cutoff, train_per_kpt, train_missed = compute_errors("train")
# train_val = model.val(split="train")
# train_mAP = train_val.pose.map50_95
# train_mAR = train_val.pose.map

# # ----------------------------
# # Compute metrics for test
# # ----------------------------
# test_errors, test_rmse, test_rmse_cutoff, test_per_kpt, test_missed = compute_errors("test")
# test_val = model.val(split="test")
# test_mAP = test_val.pose.map50_95
# test_mAR = test_val.pose.map

# # ----------------------------
# # Print Summary
# # ----------------------------
# print("\n===== Pose Evaluation Summary =====\n")
# print("| Metric             | Train       | Test        |")
# print("|-------------------|------------|------------|")
# print(f"| RMSE               | {train_rmse:10.2f} | {test_rmse:10.2f} |")
# print(f"| RMSE (cutoff 50px) | {train_rmse_cutoff:10.2f} | {test_rmse_cutoff:10.2f} |")
# print(f"| mAP                | {train_mAP:10.4f} | {test_mAP:10.4f} |")
# print(f"| mAR                | {train_mAR:10.4f} | {test_mAR:10.4f} |")

# print(f"\nTrain images skipped: {len(train_missed)}")
# print(f"Test images skipped: {len(test_missed)}\n")

# # ----------------------------
# # Print per-keypoint mean pixel errors
# # ----------------------------
# print("Per-keypoint Mean Pixel Error (Train):")
# for name, e in zip(keypoint_names, train_per_kpt):
#     print(f"{name:15}: {e:.2f} px")

# print("\nPer-keypoint Mean Pixel Error (Test):")
# for name, e in zip(keypoint_names, test_per_kpt):
#     print(f"{name:15}: {e:.2f} px")

import numpy as np
from ultralytics import YOLO
from pathlib import Path

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
model_path = "/home/nathaniel/Documents/GitHub/ClassificationTrainer/runs/pose/train10/weights/best.pt"

dataset_root = Path("/home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset")
images_base = dataset_root / "images"
labels_base = dataset_root / "labels"

pixel_cutoff = 50  # used for rmse_pcutoff
# -------------------------------------------------

model = YOLO(model_path)

keypoint_names = [
    "Nose", "Ear_left", "Ear_right", "Spine1", "Shoulder_left", "Shoulder_right",
    "Center", "Spine2", "Hip_left", "Hip_right", "Tail_base"
]

# -------------------------------------------------
# Compute pixel errors + RMSE
# -------------------------------------------------
def compute_errors(split):

    images_dir = images_base / split
    labels_dir = labels_base / split

    all_errors = []
    missed_images = []

    for img_path in sorted(images_dir.glob("*")):

        result = model.predict(img_path, verbose=False)[0]

        # Skip if no detection
        if result.keypoints is None or len(result.keypoints.xy) == 0:
            missed_images.append(img_path.name)
            continue

        pred = result.keypoints.xy[0].cpu().numpy()

        label_path = labels_dir / (img_path.stem + ".txt")
        try:
            vals = np.loadtxt(label_path)
        except Exception:
            missed_images.append(img_path.name)
            continue

        if vals.ndim > 1:
            vals = vals[0]

        kpts = vals[5:].reshape(-1,3)[:,:2]

        h, w = result.orig_shape
        gt = kpts.copy()
        gt[:,0] *= w
        gt[:,1] *= h

        dist = np.linalg.norm(pred - gt, axis=1)
        all_errors.append(dist)

    all_errors = np.array(all_errors)

    rmse = np.sqrt((all_errors**2).mean())

    mask = all_errors < pixel_cutoff
    rmse_cutoff = (
        np.sqrt((all_errors[mask]**2).mean())
        if np.any(mask) else np.nan
    )

    per_kpt_mean = all_errors.mean(axis=0)

    return all_errors, rmse, rmse_cutoff, per_kpt_mean, missed_images


# -------------------------------------------------
# Compute YOLO mAP / mAR safely across versions
# -------------------------------------------------
def compute_map_mar(split):

    val = model.val(split=split)

    # Version-agnostic extraction
    mAP = np.mean(val.pose.all_ap)
    mAR = np.mean(val.pose.r)

    return mAP, mAR


# -------------------------------------------------
# RUN TRAIN METRICS
# -------------------------------------------------
train_errors, train_rmse, train_rmse_cutoff, train_per_kpt, train_missed = compute_errors("train")
train_mAP, train_mAR = compute_map_mar("train")

# -------------------------------------------------
# RUN TEST METRICS
# -------------------------------------------------
test_errors, test_rmse, test_rmse_cutoff, test_per_kpt, test_missed = compute_errors("test")
test_mAP, test_mAR = compute_map_mar("test")

# -------------------------------------------------
# PRINT SUMMARY TABLE
# -------------------------------------------------
print("\n================ SUMMARY ================\n")

print("| Metric            | Train        | Test         |")
print("|------------------|-------------|-------------|")
print(f"| RMSE              | {train_rmse:11.3f} | {test_rmse:11.3f} |")
print(f"| RMSE_pcutoff      | {train_rmse_cutoff:11.3f} | {test_rmse_cutoff:11.3f} |")
print(f"| mAP               | {train_mAP:11.5f} | {test_mAP:11.5f} |")
print(f"| mAR               | {train_mAR:11.5f} | {test_mAR:11.5f} |")

print("\nTrain images skipped:", len(train_missed))
print("Test images skipped:", len(test_missed))

# -------------------------------------------------
# PER KEYPOINT ERRORS
# -------------------------------------------------
print("\n===== Per-Keypoint Mean Pixel Error (Train) =====")
for name, e in zip(keypoint_names, train_per_kpt):
    print(f"{name:15}: {e:7.2f} px")

print("\n===== Per-Keypoint Mean Pixel Error (Test) =====")
for name, e in zip(keypoint_names, test_per_kpt):
    print(f"{name:15}: {e:7.2f} px")

Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6422.7±502.0 MB/s, size: 1181.3 KB)
val: Scanning /home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/labels/train.cache... 439 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 439/439 1.1Mit/s 0.0s0s


/home/nathaniel/Documents/GitHub/visionpipe/visionpipe/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 1.4it/s 19.9s0.7ss
                   all        439        439      0.985      0.984      0.995      0.971      0.978      0.977      0.992      0.905
Speed: 0.4ms preprocess, 16.6ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /home/nathaniel/Documents/GitHub/ClassificationTrainer/runs/pose/val9
Ultralytics 8.3.236 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6614.8±793.7 MB/s, size: 1382.2 KB)
val: Scanning /home/nathaniel/Documents/GitHub/visionpipe/src/deep_lab_cut/YOLOv11/training_dataset/labels/test.cache... 56 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 56/56 186.1Kit/s 0.0s


/home/nathaniel/Documents/GitHub/visionpipe/visionpipe/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.4it/s 2.8s1.2s
                   all         56         56      0.911      0.909      0.961       0.82      0.928      0.927      0.958      0.814
Speed: 0.6ms preprocess, 18.4ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /home/nathaniel/Documents/GitHub/ClassificationTrainer/runs/pose/val10

================ SUMMARY ================

| Metric            | Train        | Test         |
|------------------|-------------|-------------|
| RMSE              |     282.159 |     304.618 |
| RMSE_pcutoff      |      20.096 |      23.679 |
| mAP               |     0.90548 |     0.81406 |
| mAR               |     0.97722 |     0.92696 |

Train images skipped: 0
Test images skipped: 1

===== Per-Keypoint Mean Pixel Error (Train) =====
Nose           :   37.82 px
Ear_left       :  109.07 px
Ear_right      :  108.6